# B2-022-probabilistic-latent-models — Practice p01 — Solution

**Type:** mc-normal-form · **Difficulty:** intro · **Concepts:** multivariate-gaussian

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

Let $v=x-\mu=(3-1,\,-1-0)=(2,-1)$.

**Determinant.** $\det\Sigma=2\cdot3-1\cdot1=5$.

**Solve.** For a $2\times2$ matrix, $\Sigma^{-1}=\frac1{\det\Sigma}\begin{pmatrix}3&-1\\-1&2\end{pmatrix}$, so
$$w=\Sigma^{-1}v=\tfrac15\begin{pmatrix}3\cdot2+(-1)(-1)\\(-1)\cdot2+2\cdot(-1)\end{pmatrix}=\tfrac15\begin{pmatrix}7\\-4\end{pmatrix}.$$
Check: $\Sigma w=\tfrac15(14-4,\;7-12)=(2,-1)=v$.

**Quadratic form.** $v^\top w=\tfrac15\big(2\cdot7+(-1)(-4)\big)=\tfrac{18}{5}$.

**Factor $-\tfrac12$.** $-\tfrac12\cdot\tfrac{18}{5}=-\tfrac{9}{5}$.

**Lowest terms and sign convention.** $\gcd(9,5)=1$, and the minus sign belongs to the numerator with a positive denominator: $a=-9$, $b=5$.

So $a+b=-9+5=-4$.

**Answer: C.**
(B $=-8$ would come from not reducing $-18/10$; D $=14$ from dropping the sign and using $9+5$; E $=23$ from the unhalved $18/5$.)

In [ ]:
from fractions import Fraction
from math import gcd

import torch

mu = (1, 0)
Sigma = ((2, 1), (1, 3))
x = (3, -1)
v = (x[0] - mu[0], x[1] - mu[1])
det = Sigma[0][0] * Sigma[1][1] - Sigma[0][1] * Sigma[1][0]
# Adjugate solve: Sigma^{-1} v = adj(Sigma) v / det
# Fraction: exact rational number
w = (Fraction(Sigma[1][1] * v[0] - Sigma[0][1] * v[1], det),
     Fraction(-Sigma[1][0] * v[0] + Sigma[0][0] * v[1], det))
quad = v[0] * w[0] + v[1] * w[1]
term = -Fraction(1, 2) * quad
a, b = term.numerator, term.denominator   # Fraction stores lowest terms with b > 0
ANSWER = "C"

# Floating-point cross-check with a linear solve (no explicit inverse)
S = torch.tensor(Sigma, dtype=torch.float64)
vt = torch.tensor(v, dtype=torch.float64)
quad_float = (vt @ torch.linalg.solve(S, vt)).item()
print(det, w, quad, term, a + b, quad_float)

### Answer check

In [ ]:
# Checks: the Gaussian answers against pinned values
assert det == 5
assert w == (Fraction(7, 5), Fraction(-4, 5))
assert quad == Fraction(18, 5)
assert term == Fraction(-9, 5)
assert (a, b) == (-9, 5) and gcd(a, b) == 1 and b > 0
assert a + b == -4
assert abs(quad_float - 3.6) < 1e-12
assert ANSWER == "C"